In [ ]:
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.timeseries import LombScargle
import astropy.units as u
import gyrointerp
from gyrointerp import gyro_age_posterior
from gyrointerp import get_summary_statistics

targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\ASTR502_Mega_Target_List.csv")

In [ ]:
k2_targets = pd.read("C:\\Users\\smithlt\\Documents\\ASTR502\\K2_data.txt")

print(k2_targets.head())

In [ ]:
#create an array of the K2 target TIC IDs
k2_target_tic_ids = k2_targets['tic_id'].values

In [ ]:
if 'mission_source' in targets.columns:
    print(f"\nMission sources in dataset:")
    print(targets['mission_source'].value_counts())
    
    # Filter for only K2 targets
    k2_targets = targets[targets['mission_source'] == 'K2'].copy()
    print(f"\nFound {len(k2_targets)} K2 targets!")

    #Filter for K2 targets with TIC IDs that match k2_target_tic_ids
    k2_targets = k2_targets[k2_targets['tic_id'].isin(k2_target_tic_ids)]
    if len(k2_targets) > 0:
        print("\nK2 targets:")
        for i, row in k2_targets.head().iterrows():
            print(f"  {row['pl_name']}: {row['tic_id']}")
    else:
        print("No K2 targets found in the dataset.")
        print("Let's check for targets that might have K2 data...")
        
        # Look for targets that might have sy_kepmag (Kepler magnitude)
        if 'sy_kepmag' in targets.columns:
            k2_mag_targets = targets[pd.notna(targets['sy_kepmag'])].copy()
            print(f"Found {len(k2_mag_targets)} targets with K2 magnitudes")
            if len(k2_mag_targets) > 0:
                print("Targets with matching K2 TIC IDs:")
                for i, row in k2_mag_targets.head().iterrows():
                    print(f"  {row['pl_name']}: {row['tic_id']}, K2 mag: {row['sy_kepmag']}")
                k2_targets = k2_mag_targets
else:
    print("No 'mission_source' column found. Let's look for other K2 indicators...")

    # Look for targets that might have sy_kepmag
    if 'sy_kepmag' in targets.columns:
        k2_targets = targets[pd.notna(targets['sy_kepmag'])].copy()
        print(f"Found {len(k2_targets)} targets with K2 magnitudes")
    else:
        print("No clear K2 indicators found in the dataset.")
        k2_targets = pd.DataFrame()

print(f"\nFinal K2 target count: {len(k2_targets)}")

if len(k2_targets) > 0:
    working_k2_targets = []

    if working_k2_targets:
            # DataFrame for processing
            # Add the temperature data from st_teff column

            #create a dataframe to store results that contains the TIC ID and period from k2_targets, and the target name and temperature from targets
            target_results = []
            for i, row in k2_targets.iterrows():
                target_name = row['pl_name']
                tic_id = row['tic_id']
                mission_name = row['mission_source']
                clean_time = row['time']
                teff = row['st_teff']

                target_results.append({
                    'target_name': target_name,
                    'tic_id': tic_id,
                    'mission': mission_name,
                    'data_points': len(clean_time),
                    'Teff': teff
                })
            k2_star_df = pd.DataFrame(target_results)
            print("\nReady to process these K2 targets:")
            for i, row in k2_star_df.iterrows():
                print(f"  {row['target_name']}: {row['tic_id']} ({row['data_points']} data points, Teff: {row['Teff']})")

In [ ]:
Teff = k2_star_df['Teff']
Prot = k2_star_df['period']

In [ ]:
#cycle through all of the 10 targets and run gyrointerp
#print the age posterior calculated for each target

for i in range(len(k2_star_df)):

    # units: days
    Prot, Prot_err = k2_star_df['period'].iloc[i], 0.2
    print(Prot)

    # units: kelvin
    Teff, Teff_err = k2_star_df['Teff'].iloc[i], 100
    print(Teff)

    # uniformly spaced grid between 0 and 4000 megayears
    age_grid = np.linspace(0, 4000, 500)

    # calculate the age posterior at each age in `age_grid`
    age_posterior = gyro_age_posterior(
        Prot, Teff,
        Prot_err=Prot_err, Teff_err=Teff_err,
        age_grid=age_grid
    )

    print(f"\nTarget: {k2_star_df['target_name'].iloc[i]}")
    print(f"Age posterior (normalized): {age_posterior / np.sum(age_posterior)}")
    print(f"Age (most probable): {age_grid[np.argmax(age_posterior)]:.1f} Myr")